# 🖼️ Bài Thực Hành: Canny Edge Detector
**Môn học:** Xử lý ảnh và Thị giác máy tính  
**Ảnh đầu vào:** `in.jpg`

---

> Trong bài này mình sẽ tìm hiểu và cài đặt thuật toán **Canny Edge Detection** – một trong những thuật toán phát hiện cạnh phổ biến nhất trong xử lý ảnh. Bài gồm 3 phần chính: Lý thuyết → Thực hành → Câu hỏi mở rộng.

---
# 📌 PHẦN I: LÝ THUYẾT
---

## 1. Các bước của thuật toán Canny

Thuật toán Canny được John F. Canny đề xuất năm 1986, gồm **5 bước chính**:

---

### Bước 1: Giảm nhiễu – Gaussian Blur

Trước khi tính gradient, ta cần làm mịn ảnh để loại bỏ nhiễu nhỏ (noise) có thể tạo ra cạnh giả.

- Sử dụng **bộ lọc Gaussian** (Gaussian kernel) tích chập với ảnh.
- Tham số **sigma (σ)** điều chỉnh mức độ làm mịn: sigma càng lớn → làm mịn càng nhiều → mất chi tiết nhỏ.

$$G(x,y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$$

---

### Bước 2: Tính Gradient – Sobel Filter

Sau khi làm mịn, ta tính **gradient** theo hai hướng x và y dùng toán tử Sobel:

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad G_y = \begin{bmatrix} 1 & 2 & 1 \\ 0 & 0 & 0 \\ -1 & -2 & -1 \end{bmatrix}$$

- **Độ lớn gradient:** $G = \sqrt{G_x^2 + G_y^2}$
- **Hướng gradient:** $\theta = \arctan\left(\frac{G_y}{G_x}\right)$

Nơi nào gradient lớn → có sự thay đổi cường độ mạnh → đó là cạnh.

---

### Bước 3: Non-Maximum Suppression (NMS)

Sau bước 2, vùng cạnh thường bị "dày" (nhiều pixel liền nhau đều có gradient cao). Bước này **làm mỏng cạnh** bằng cách:

- Với mỗi pixel, kiểm tra 2 hàng xóm theo **hướng gradient**.
- Nếu pixel hiện tại **không phải cực đại cục bộ** → đặt về 0 (loại bỏ).
- Kết quả: cạnh chỉ còn dày **1 pixel**.

---

### Bước 4: Ngưỡng kép – Double Thresholding

Dùng **hai ngưỡng** T_low và T_high để phân loại pixel:

| Điều kiện | Phân loại |
|---|---|
| G > T_high | **Strong edge** (cạnh mạnh) |
| T_low < G ≤ T_high | **Weak edge** (cạnh yếu) |
| G ≤ T_low | Không phải cạnh → loại bỏ |

---

### Bước 5: Edge Tracking by Hysteresis

Bước cuối cùng quyết định xem **weak edge** có được giữ lại không:

- Nếu weak edge **kết nối với strong edge** (8-connected) → giữ lại.
- Nếu weak edge **đứng một mình**, không kết nối → loại bỏ (có thể là nhiễu).

➡️ Kết quả cuối: ảnh cạnh rõ ràng, liên tục, ít nhiễu.

## 2. Các tham số và ảnh hưởng

### 🔹 Sigma (σ) – Độ làm mịn Gaussian

| Sigma nhỏ | Sigma lớn |
|---|---|
| Làm mịn ít, giữ nhiều chi tiết | Làm mịn nhiều, mất chi tiết nhỏ |
| Dễ bị ảnh hưởng bởi nhiễu | Chống nhiễu tốt hơn |
| Phát hiện nhiều cạnh nhỏ | Chỉ phát hiện cạnh lớn, rõ |

→ **Thường dùng:** σ = 1.0 đến 2.0

---

### 🔹 Threshold thấp (T_low)

- Ngưỡng dưới để xác định **weak edge**.
- T_low **càng nhỏ** → nhiều pixel được coi là weak edge → kết quả nhiều cạnh hơn (có thể lẫn nhiễu).
- T_low **càng lớn** → ít weak edge → cạnh bị thiếu, đứt đoạn.

---

### 🔹 Threshold cao (T_high)

- Ngưỡng trên để xác định **strong edge**.
- T_high **càng cao** → chỉ giữ cạnh rất mạnh → cạnh sắc nét nhưng có thể bị thiếu.
- T_high **càng thấp** → nhiều cạnh mạnh hơn → dễ lẫn nhiễu.

> 💡 **Kinh nghiệm thực hành:** Tỷ lệ T_high / T_low ≈ 2:1 hoặc 3:1 thường cho kết quả tốt.

## 3. Ưu và nhược điểm của Canny

### a. So sánh với Sobel và Laplacian

| Tiêu chí | Canny | Sobel | Laplacian |
|---|---|---|---|
| Độ chính xác | ⭐⭐⭐⭐⭐ Rất cao | ⭐⭐⭐ Trung bình | ⭐⭐⭐ Trung bình |
| Tốc độ | ⭐⭐⭐ Chậm hơn | ⭐⭐⭐⭐⭐ Nhanh | ⭐⭐⭐⭐⭐ Nhanh |
| Chống nhiễu | ⭐⭐⭐⭐⭐ Tốt (có Gaussian) | ⭐⭐ Kém | ⭐ Rất kém |
| Cạnh mỏng | ✅ Có (NMS) | ❌ Không | ❌ Không |
| Tham số điều chỉnh | Nhiều | Ít | Rất ít |

---

### b. Ưu điểm
- ✅ Phát hiện cạnh **chính xác**, mỏng (1 pixel)
- ✅ **Chống nhiễu tốt** nhờ Gaussian blur
- ✅ Ít cạnh giả nhờ double threshold + hysteresis
- ✅ Có thể điều chỉnh linh hoạt qua tham số

### b. Nhược điểm
- ❌ **Chậm hơn** Sobel/Laplacian
- ❌ Cần chọn tham số cẩn thận
- ❌ Gaussian blur có thể làm mờ các cạnh gần nhau
- ❌ Không hoạt động tốt trên ảnh có độ tương phản rất thấp

---

### c. Ứng dụng thực tế

1. 🚗 **Xe tự hành:** Phát hiện làn đường, biển báo, chướng ngại vật
2. 🏥 **Y tế:** Phân tích ảnh X-quang, MRI – tìm ranh giới mô, khối u
3. 📷 **Nhận dạng vật thể:** Tìm contour để nhận dạng hình dạng sản phẩm
4. 🛡️ **Camera an ninh:** Phát hiện chuyển động, nhận dạng khuôn mặt
5. 🗺️ **Bản đồ vệ tinh:** Phát hiện đường, sông, biên giới công trình

---
# 📌 PHẦN II: THỰC HÀNH
---

## 1. Cài đặt thư viện và đọc ảnh

In [1]:
# Import các thư viện cần thiết
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage import feature, filters, exposure
from skimage.util import random_noise
from scipy.ndimage import gaussian_filter

print("✅ Import thư viện thành công!")
print(f"   OpenCV version: {cv2.__version__}")
print(f"   NumPy version: {np.__version__}")

ModuleNotFoundError: No module named 'skimage'

In [ ]:
# ========================
# Đọc ảnh và chuyển grayscale
# ========================

# Đọc ảnh màu bằng OpenCV (BGR format)
img_bgr = cv2.imread('in.jpg')

if img_bgr is None:
    raise FileNotFoundError("❌ Không tìm thấy file in.jpg! Hãy chắc chắn file nằm cùng thư mục với notebook.")

# Chuyển BGR → RGB để hiển thị đúng màu
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Chuyển sang grayscale
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print(f"✅ Đọc ảnh thành công!")
print(f"   Kích thước ảnh màu: {img_rgb.shape}")
print(f"   Kích thước ảnh grayscale: {img_gray.shape}")
print(f"   Kiểu dữ liệu: {img_gray.dtype}")

In [ ]:
# Hiển thị ảnh gốc và grayscale
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Ảnh gốc (RGB)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title('Ảnh Grayscale', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Ảnh Đầu Vào', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output_1_input.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Hiển thị ảnh đầu vào xong.")

## 2. Áp dụng Canny

### 2a. Dùng OpenCV – `cv2.Canny`

`cv2.Canny(image, threshold1, threshold2, apertureSize=3, L2gradient=False)`
- `threshold1`: ngưỡng thấp
- `threshold2`: ngưỡng cao
- `apertureSize`: kích thước kernel Sobel (thường = 3)

> ⚠️ Lưu ý: `cv2.Canny` tích hợp sẵn Gaussian blur bên trong, nhưng mình có thể blur thủ công trước để kiểm soát tốt hơn.

In [ ]:
# ========================
# Áp dụng Gaussian Blur rồi dùng cv2.Canny
# ========================

# Bước 1: Làm mịn ảnh
img_blurred = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.0)

# Bước 2: Áp dụng Canny với tham số mặc định
canny_default = cv2.Canny(img_blurred, threshold1=50, threshold2=150)

# Hiển thị kết quả
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Ảnh Grayscale gốc', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_blurred, cmap='gray')
axes[1].set_title('Sau Gaussian Blur (σ=1.0)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(canny_default, cmap='gray')
axes[2].set_title('Canny Edge (T_low=50, T_high=150)', fontsize=12)
axes[2].axis('off')

plt.suptitle('OpenCV Canny – Tham số mặc định', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_2_canny_default.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Canny OpenCV tham số mặc định xong.")

### 2b. Dùng Scikit-image – `skimage.feature.canny`

`skimage.feature.canny(image, sigma=1.0, low_threshold=None, high_threshold=None)`

Điểm khác biệt so với OpenCV:
- Input là float [0, 1], output là boolean array
- Tham số `sigma` tích hợp thẳng vào hàm
- Có thể tự động tính ngưỡng nếu không truyền vào

In [ ]:
# ========================
# Áp dụng Canny bằng Scikit-image
# ========================

# Chuẩn hóa ảnh về [0, 1] cho skimage
img_norm = img_gray.astype(float) / 255.0

# Canny skimage – tự động ngưỡng
canny_ski_auto = feature.canny(img_norm, sigma=1.0)

# Canny skimage – tham số tương đương OpenCV
canny_ski_manual = feature.canny(img_norm, sigma=1.0,
                                  low_threshold=50/255,
                                  high_threshold=150/255)

# So sánh OpenCV vs Scikit-image
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(canny_default, cmap='gray')
axes[0].set_title('OpenCV Canny\n(T_low=50, T_high=150)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(canny_ski_auto, cmap='gray')
axes[1].set_title('Skimage Canny\n(Tự động ngưỡng, σ=1.0)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(canny_ski_manual, cmap='gray')
axes[2].set_title('Skimage Canny\n(T_low=50/255, T_high=150/255)', fontsize=12)
axes[2].axis('off')

plt.suptitle('So sánh OpenCV vs Scikit-image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_3_opencv_vs_skimage.png', dpi=100, bbox_inches='tight')
plt.show()

# Đếm số pixel cạnh
print(f"📊 Số pixel cạnh – OpenCV: {np.sum(canny_default > 0)}")
print(f"📊 Số pixel cạnh – Skimage (auto): {np.sum(canny_ski_auto)}")
print(f"📊 Số pixel cạnh – Skimage (manual): {np.sum(canny_ski_manual)}")

**📝 Nhận xét:**
- OpenCV và Skimage cho kết quả tương tự nhau khi dùng cùng ngưỡng.
- Skimage với tự động ngưỡng thường phát hiện nhiều cạnh hơn một chút vì nó phân tích histogram ảnh.
- OpenCV nhanh hơn và phù hợp cho ứng dụng real-time; Skimage linh hoạt hơn cho nghiên cứu.

## 3. Thay đổi tham số

Bây giờ mình sẽ thử nhiều bộ tham số khác nhau để thấy rõ ảnh hưởng của từng tham số lên kết quả.

In [ ]:
# ========================
# Thay đổi Sigma
# ========================

sigma_values = [0.5, 1.0, 2.0, 3.0]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, sigma in enumerate(sigma_values):
    result = feature.canny(img_norm, sigma=sigma,
                           low_threshold=50/255,
                           high_threshold=150/255)
    axes[i].imshow(result, cmap='gray')
    axes[i].set_title(f'σ = {sigma}\nSố cạnh: {np.sum(result)}', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Ảnh hưởng của Sigma (T_low=50, T_high=150)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_4_sigma.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Thay đổi sigma xong.")

**📝 Nhận xét thay đổi Sigma:**
- σ nhỏ (0.5): Giữ nhiều chi tiết → phát hiện cạnh nhỏ, nhưng cũng bắt nhiều nhiễu hơn
- σ = 1.0–2.0: Cân bằng tốt giữa chi tiết và chống nhiễu – thường dùng nhất
- σ lớn (3.0): Chỉ còn cạnh lớn, rõ ràng → mất chi tiết nhỏ

In [ ]:
# ========================
# Thay đổi Threshold (OpenCV)
# ========================

# 3 bộ tham số threshold khác nhau
param_sets = [
    {'t_low': 20,  't_high': 60,  'label': 'Ngưỡng thấp\n(T_low=20, T_high=60)'},
    {'t_low': 50,  't_high': 150, 'label': 'Ngưỡng trung bình\n(T_low=50, T_high=150)'},
    {'t_low': 100, 't_high': 200, 'label': 'Ngưỡng cao\n(T_low=100, T_high=200)'},
    {'t_low': 30,  't_high': 90,  'label': 'Tỷ lệ 1:3\n(T_low=30, T_high=90)'},
]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for i, p in enumerate(param_sets):
    blurred = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.0)
    result = cv2.Canny(blurred, p['t_low'], p['t_high'])
    n_edges = np.sum(result > 0)
    axes[i].imshow(result, cmap='gray')
    axes[i].set_title(f"{p['label']}\nSố cạnh: {n_edges}", fontsize=10)
    axes[i].axis('off')

plt.suptitle('Ảnh hưởng của Threshold (σ=1.0)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_5_threshold.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Thay đổi threshold xong.")

**📝 Nhận xét thay đổi Threshold:**
- **Ngưỡng thấp (20/60):** Phát hiện được rất nhiều cạnh, kể cả cạnh yếu → ảnh nhiều chi tiết nhưng cũng nhiều nhiễu
- **Ngưỡng trung bình (50/150):** Cân bằng – cạnh rõ ràng, ít nhiễu – thường dùng trong thực tế
- **Ngưỡng cao (100/200):** Chỉ giữ cạnh mạnh nhất → ảnh sạch nhưng có thể bị thiếu cạnh quan trọng
- **Tỷ lệ 1:3 (30/90):** Tỷ lệ T_high/T_low = 3 được khuyến nghị bởi Canny gốc

In [ ]:
# ========================
# Tổng hợp: Thay đổi cả sigma và threshold cùng lúc
# ========================

combos = [
    (0.5, 30/255, 90/255,  'σ=0.5, T=30/90'),
    (1.0, 50/255, 150/255, 'σ=1.0, T=50/150'),
    (2.0, 50/255, 150/255, 'σ=2.0, T=50/150'),
    (1.5, 20/255, 60/255,  'σ=1.5, T=20/60'),
    (1.0, 80/255, 200/255, 'σ=1.0, T=80/200'),
    (3.0, 30/255, 90/255,  'σ=3.0, T=30/90'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (sig, tl, th, title) in enumerate(combos):
    result = feature.canny(img_norm, sigma=sig,
                           low_threshold=tl, high_threshold=th)
    axes[i].imshow(result, cmap='gray')
    axes[i].set_title(f'{title}\nSố pixel cạnh: {np.sum(result)}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('So sánh 6 bộ tham số khác nhau', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_6_param_compare.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 So sánh 6 bộ tham số xong.")

## 4. Áp dụng với các loại ảnh khác nhau

Mình sẽ tạo 3 phiên bản từ `in.jpg` để kiểm tra khả năng hoạt động của Canny trong các điều kiện khác nhau.

In [ ]:
# ========================
# Tạo các phiên bản ảnh biến thể
# ========================

# 1. Ảnh nhiễu Gaussian
img_noisy = random_noise(img_norm, mode='gaussian', var=0.02)
img_noisy = np.clip(img_noisy, 0, 1)

# 2. Ảnh độ tương phản thấp
img_low_contrast = exposure.rescale_intensity(img_norm, in_range=(0.2, 0.8))

# 3. Ảnh nhiều chi tiết (tăng độ sắc nét – sharpen)
blur_for_sharpen = gaussian_filter(img_norm, sigma=1)
img_detail = np.clip(img_norm + 1.5 * (img_norm - blur_for_sharpen), 0, 1)

# Hiển thị 3 phiên bản
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, img, title in zip(axes,
    [img_norm, img_noisy, img_low_contrast, img_detail],
    ['Gốc', 'Ảnh nhiễu', 'Tương phản thấp', 'Nhiều chi tiết']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Các phiên bản ảnh biến thể', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_7_variants.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Tạo ảnh biến thể xong.")

In [ ]:
# ========================
# Áp dụng Canny cho từng loại ảnh
# ========================

sigma = 1.5
t_low, t_high = 30/255, 90/255

variants = [
    (img_norm,         'Ảnh gốc'),
    (img_noisy,        'Ảnh nhiễu (Gaussian noise)'),
    (img_low_contrast, 'Tương phản thấp'),
    (img_detail,       'Nhiều chi tiết (sharpened)'),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for i, (img, title) in enumerate(variants):
    # Hàng trên: ảnh gốc biến thể
    axes[0][i].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0][i].set_title(title, fontsize=11)
    axes[0][i].axis('off')

    # Hàng dưới: kết quả Canny
    edges = feature.canny(img, sigma=sigma, low_threshold=t_low, high_threshold=t_high)
    axes[1][i].imshow(edges, cmap='gray')
    axes[1][i].set_title(f'Canny – {np.sum(edges)} px cạnh', fontsize=11)
    axes[1][i].axis('off')

axes[0][0].set_ylabel('Ảnh đầu vào', fontsize=12, labelpad=10)
axes[1][0].set_ylabel('Kết quả Canny', fontsize=12, labelpad=10)

plt.suptitle(f'Canny trên các loại ảnh (σ={sigma}, T_low={t_low:.2f}, T_high={t_high:.2f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output_8_canny_variants.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Áp dụng Canny trên các phiên bản ảnh xong.")

**📝 Nhận xét kết quả trên các loại ảnh:**

| Loại ảnh | Nhận xét |
|---|---|
| **Ảnh gốc** | Cạnh rõ ràng, đúng vị trí |
| **Ảnh nhiễu** | Canny vẫn phát hiện được nhờ Gaussian blur trước đó, nhưng có thêm một số cạnh giả do nhiễu |
| **Tương phản thấp** | Khó phân biệt cạnh thật, ít cạnh được phát hiện hơn |
| **Nhiều chi tiết** | Phát hiện được nhiều cạnh nhỏ, chi tiết hơn – nhưng có thể bị rối nếu ngưỡng thấp quá |

## 5. Kết hợp Canny với kỹ thuật khác

### 5a. Kết hợp Canny + Contour Detection

Dùng kết quả Canny làm đầu vào cho `cv2.findContours` để tìm đường viền các vật thể.

In [ ]:
# ========================
# Kết hợp Canny + Contour Detection
# ========================

# Cạnh canny dùng OpenCV
blurred = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.0)
edges_for_contour = cv2.Canny(blurred, 50, 150)

# Tìm contours từ kết quả Canny
contours, hierarchy = cv2.findContours(
    edges_for_contour.copy(),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Lọc các contour có diện tích > ngưỡng (bỏ nhiễu nhỏ)
min_area = 200
significant_contours = [c for c in contours if cv2.contourArea(c) > min_area]

# Vẽ contours lên ảnh màu
img_contour = img_rgb.copy()
cv2.drawContours(img_contour, significant_contours, -1, (0, 255, 0), 2)

# Hiển thị
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Ảnh gốc', fontsize=12)
axes[0].axis('off')

axes[1].imshow(edges_for_contour, cmap='gray')
axes[1].set_title('Canny Edge', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img_contour)
axes[2].set_title(f'Contours được phát hiện\n({len(significant_contours)} đường viền)', fontsize=12)
axes[2].axis('off')

plt.suptitle('Canny + Contour Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_9_contour.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Tổng số contour: {len(contours)}")
print(f"✅ Contour đáng kể (area > {min_area}): {len(significant_contours)}")

### 5b. Kết hợp Canny + Hough Transform (tìm đường thẳng)

In [ ]:
# ========================
# Kết hợp Canny + Hough Lines
# ========================

edges_hough = cv2.Canny(blurred, 50, 150)

# Hough Transform probabilistic – tìm đoạn thẳng
lines = cv2.HoughLinesP(
    edges_hough,
    rho=1,
    theta=np.pi/180,
    threshold=80,
    minLineLength=50,
    maxLineGap=10
)

# Vẽ đường thẳng lên ảnh
img_lines = img_rgb.copy()
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(img_lines, (x1, y1), (x2, y2), (255, 0, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Ảnh gốc', fontsize=12)
axes[0].axis('off')

axes[1].imshow(edges_hough, cmap='gray')
axes[1].set_title('Canny Edge', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img_lines)
n_lines = len(lines) if lines is not None else 0
axes[2].set_title(f'Đường thẳng phát hiện bởi Hough\n({n_lines} đoạn thẳng)', fontsize=12)
axes[2].axis('off')

plt.suptitle('Canny + Hough Transform (Line Detection)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_10_hough.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Số đường thẳng phát hiện: {n_lines}")

### 5c. Kết hợp Canny + Phân đoạn ảnh (Thresholding)

In [ ]:
# ========================
# Kết hợp Canny + Otsu Threshold
# ========================

# Phân đoạn bằng Otsu
_, otsu_mask = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Lấy cạnh Canny
edges_combine = cv2.Canny(blurred, 50, 150)

# Kết hợp: chỉ giữ cạnh nằm trong vùng tiền cảnh (foreground)
edges_masked = cv2.bitwise_and(edges_combine, otsu_mask)

# Overlay cạnh lên ảnh gốc
img_overlay = img_rgb.copy()
img_overlay[edges_combine > 0] = [255, 0, 0]  # màu đỏ cho tất cả cạnh
img_overlay[edges_masked > 0] = [0, 255, 0]   # màu xanh cho cạnh trong foreground

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Grayscale', fontsize=11)
axes[0].axis('off')

axes[1].imshow(otsu_mask, cmap='gray')
axes[1].set_title('Otsu Threshold\n(Phân đoạn)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(edges_masked, cmap='gray')
axes[2].set_title('Cạnh trong foreground\n(Canny ∩ Otsu mask)', fontsize=11)
axes[2].axis('off')

axes[3].imshow(img_overlay)
axes[3].set_title('Overlay: đỏ=tất cả cạnh\nxanh=cạnh foreground', fontsize=11)
axes[3].axis('off')

plt.suptitle('Canny + Otsu Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_11_threshold_combine.png', dpi=100, bbox_inches='tight')
plt.show()
print("📌 Kết hợp Canny + Otsu xong.")

**📝 Nhận xét phần kết hợp kỹ thuật:**
- **Canny + Contour:** Rất hữu ích để đếm/định vị vật thể trong ảnh
- **Canny + Hough:** Tìm các đường thẳng, ứng dụng tốt trong phát hiện làn đường xe
- **Canny + Otsu:** Lọc bỏ cạnh nằm trong vùng nền, chỉ giữ cạnh có ý nghĩa

---
# 📌 PHẦN III: CÂU HỎI MỞ RỘNG
---

## Câu 1: Làm thế nào để đánh giá chất lượng cạnh?

Đánh giá chất lượng cạnh thường dựa trên **3 tiêu chí** mà Canny đề xuất:

1. **Good detection (Phát hiện tốt):** Tỷ lệ cạnh thật được phát hiện cao (ít False Negative), tỷ lệ nhiễu bị phát hiện nhầm thấp (ít False Positive)
   - Dùng chỉ số: **Precision, Recall, F1-score** so với ground truth

2. **Good localization (Định vị tốt):** Cạnh phát hiện gần với cạnh thật
   - Dùng chỉ số: **Pratt Figure of Merit (FOM)**
   $$FOM = \frac{1}{\max(N_I, N_A)} \sum_{k=1}^{N_A} \frac{1}{1 + d_k^2 \alpha}$$
   Trong đó $d_k$ là khoảng cách từ cạnh phát hiện đến cạnh thật gần nhất.

3. **Single response (Phản hồi đơn):** Mỗi cạnh thật chỉ tạo ra một cạnh phát hiện (không bị nhân đôi)

**Trong thực tế**, nếu không có ground truth, có thể dùng:
- Đếm số pixel cạnh (ít ≠ tốt, nhiều ≠ xấu, phụ thuộc bài toán)
- Kiểm tra trực quan (xem cạnh có liên tục, đúng vị trí không)
- Dùng dataset benchmark như **BSDS500** (Berkeley Segmentation Dataset)

## Câu 2: Có cách nào cải thiện hiệu suất Canny không?

Có! Dưới đây là một số hướng cải thiện:

### Về tham số:
- **Tự động chọn ngưỡng (Auto-threshold):** Dùng phân tích histogram (như Otsu) hoặc median để tự động tính T_low, T_high:
  ```python
  v = np.median(img)
  t_low = int(max(0, 0.7 * v))
  t_high = int(min(255, 1.3 * v))
  ```
- **Bilateral Filter** thay cho Gaussian: làm mịn nhiễu nhưng vẫn giữ cạnh sắc nét hơn.

### Về thuật toán:
- **Denoising nâng cao:** Non-local Means, BM3D trước khi Canny
- **Multi-scale Canny:** Áp dụng nhiều sigma, tổng hợp kết quả → phát hiện cạnh ở nhiều tỷ lệ
- **Deep learning:** Các model như **HED (Holistically-nested Edge Detection)** hay **DEXTR** học cách phát hiện cạnh từ dữ liệu, vượt trội hơn Canny trong hầu hết trường hợp

### Về tốc độ:
- Sử dụng **CUDA/GPU** qua OpenCV-CUDA
- Giảm độ phân giải ảnh trước khi xử lý nếu không cần chi tiết cao

## Câu 3: Canny có dùng cho ảnh màu không? Nếu có thì làm sao?

**Canny truyền thống chỉ làm việc với ảnh grayscale.** Nhưng có thể áp dụng cho ảnh màu theo 2 cách:

### Cách 1: Chuyển về grayscale trước (đơn giản, phổ biến)
```python
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150)
```
- **Ưu điểm:** Nhanh, đơn giản
- **Nhược điểm:** Mất thông tin màu → có thể bỏ sót cạnh mà chỉ thấy khi phân biệt màu

### Cách 2: Color Canny – xử lý từng kênh màu (chính xác hơn)
```python
# Áp dụng Canny cho từng kênh R, G, B
edges_r = cv2.Canny(img[:,:,0], 50, 150)
edges_g = cv2.Canny(img[:,:,1], 50, 150)  
edges_b = cv2.Canny(img[:,:,2], 50, 150)
# Kết hợp: lấy OR
edges_color = np.maximum(np.maximum(edges_r, edges_g), edges_b)
```

### Cách 3: Color gradient (chuẩn nhất về mặt lý thuyết)
- Tính gradient vector (Gx, Gy) cho từng kênh màu
- Lấy gradient tổng hợp: $G = \sqrt{G_{rx}^2 + G_{gx}^2 + G_{bx}^2 + G_{ry}^2 + G_{gy}^2 + G_{by}^2}$
- Phát hiện được cạnh màu mà grayscale không thấy

In [ ]:
# ========================
# Minh họa Color Canny vs Grayscale Canny
# ========================

# Cách 1: Grayscale Canny
edges_gray = cv2.Canny(cv2.GaussianBlur(img_gray, (5,5), 1.0), 50, 150)

# Cách 2: Color Canny (OR từng kênh)
edges_per_channel = []
for ch in range(3):
    channel = img_rgb[:,:,ch]
    blurred_ch = cv2.GaussianBlur(channel, (5,5), 1.0)
    e = cv2.Canny(blurred_ch, 50, 150)
    edges_per_channel.append(e)

edges_color_or = np.maximum(np.maximum(edges_per_channel[0],
                                        edges_per_channel[1]),
                             edges_per_channel[2])

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

ch_names = ['Kênh Red', 'Kênh Green', 'Kênh Blue']
for i in range(3):
    axes[0][i].imshow(edges_per_channel[i], cmap='gray')
    axes[0][i].set_title(f'Canny – {ch_names[i]}', fontsize=11)
    axes[0][i].axis('off')

axes[1][0].imshow(edges_gray, cmap='gray')
axes[1][0].set_title('Grayscale Canny', fontsize=11)
axes[1][0].axis('off')

axes[1][1].imshow(edges_color_or, cmap='gray')
axes[1][1].set_title('Color Canny (OR 3 kênh)', fontsize=11)
axes[1][1].axis('off')

diff = edges_color_or.astype(int) - edges_gray.astype(int)
diff_vis = np.clip(diff, 0, 255).astype(np.uint8)
axes[1][2].imshow(diff_vis, cmap='hot')
axes[1][2].set_title('Cạnh thêm khi dùng Color Canny\n(so với Grayscale)', fontsize=11)
axes[1][2].axis('off')

plt.suptitle('Canny cho ảnh màu – So sánh các cách', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_12_color_canny.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"📊 Grayscale Canny: {np.sum(edges_gray > 0)} pixel cạnh")
print(f"📊 Color Canny (OR): {np.sum(edges_color_or > 0)} pixel cạnh")

## Câu 4: Áp dụng Canny cho video như thế nào?

Về mặt kỹ thuật, video chỉ là **chuỗi các frame ảnh liên tiếp**. Canny có thể áp dụng cho video bằng cách xử lý từng frame:

```python
import cv2

# Mở video
cap = cv2.VideoCapture('input.mp4')

# Thiết lập output writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('output_edges.mp4', fourcc, fps, (w, h), isColor=False)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Xử lý từng frame
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5,5), sigmaX=1.0)
    edges = cv2.Canny(blurred, 50, 150)
    
    # Ghi frame vào output
    out.write(edges)

cap.release()
out.release()
print("Done!")
```

**Một số lưu ý khi dùng Canny cho video:**

| Vấn đề | Giải pháp |
|---|---|
| Cạnh nhấp nháy giữa các frame | Dùng temporal smoothing hoặc optical flow |
| Tốc độ chậm với video HD | Resize frame trước, dùng threading/multiprocessing |
| Ảnh hưởng của chuyển động mờ | Tăng sigma để giảm nhiễu từ motion blur |
| Background cố định | Kết hợp Background Subtraction trước Canny |

---
# 🏁 KẾT LUẬN
---

## Nhận xét tổng quan về Canny

Qua bài thực hành này, mình rút ra được những điểm sau về thuật toán Canny:

### ✅ Điểm mạnh:
1. **Chính xác cao** – cạnh mỏng 1 pixel, ít cạnh giả nhờ NMS và hysteresis
2. **Chống nhiễu tốt** – Gaussian blur tích hợp từ đầu pipeline
3. **Linh hoạt** – có thể điều chỉnh sigma, threshold để phù hợp từng bài toán
4. **Được hỗ trợ rộng rãi** – có trong OpenCV, Scikit-image, dễ sử dụng

### ❌ Điểm yếu:
1. **Nhạy với tham số** – phải chọn sigma và threshold phù hợp, không phải lúc nào cũng tự động
2. **Gaussian blur làm mờ cạnh gần nhau** – đặc biệt khi sigma lớn
3. **Không đủ mạnh cho ảnh rất phức tạp** – deep learning edge detection (HED, DEXTR) vượt trội hơn

---

## Khi nào nên dùng Canny?

| Nên dùng | Không nên dùng |
|---|---|
| Ảnh có cạnh rõ ràng, tương phản đủ cao | Ảnh quá nhiễu nặng |
| Cần cạnh mỏng, liên tục, chính xác | Cần tốc độ xử lý tối đa (real-time cực nhanh) |
| Tiền xử lý cho contour/shape detection | Ảnh y tế phức tạp (MRI, CT) – cần model chuyên biệt |
| Môi trường có giới hạn tài nguyên (không dùng được deep learning) | Cần phân biệt cạnh theo ngữ nghĩa |

> 💡 **Kết luận cá nhân:** Canny vẫn là lựa chọn hàng đầu trong các bài toán xử lý ảnh truyền thống nhờ sự cân bằng tốt giữa độ chính xác và tốc độ. Dù đã ra đời từ năm 1986 nhưng đến nay vẫn được dùng rộng rãi và là nền tảng cho nhiều kỹ thuật phức tạp hơn.

In [ ]:
# ========================
# Summary: Hiển thị tổng kết tất cả kết quả chính
# ========================

fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.3)

# Row 1: Input
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(img_rgb)
ax1.set_title('1. Ảnh gốc', fontsize=10)
ax1.axis('off')

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(img_gray, cmap='gray')
ax2.set_title('2. Grayscale', fontsize=10)
ax2.axis('off')

ax3 = fig.add_subplot(gs[0, 2])
ax3.imshow(img_blurred, cmap='gray')
ax3.set_title('3. Gaussian Blur (σ=1)', fontsize=10)
ax3.axis('off')

ax4 = fig.add_subplot(gs[0, 3])
ax4.imshow(canny_default, cmap='gray')
ax4.set_title('4. Canny mặc định (50/150)', fontsize=10)
ax4.axis('off')

# Row 2: Parameter variations
sigmas_sum = [0.5, 1.5, 3.0, 1.0]
labels_sum = ['σ=0.5, T=50/150', 'σ=1.5, T=50/150', 'σ=3.0, T=50/150', 'σ=1.0, T=20/60']
thresholds_sum = [(50/255, 150/255), (50/255, 150/255), (50/255, 150/255), (20/255, 60/255)]

for col in range(4):
    ax = fig.add_subplot(gs[1, col])
    e = feature.canny(img_norm, sigma=sigmas_sum[col],
                      low_threshold=thresholds_sum[col][0],
                      high_threshold=thresholds_sum[col][1])
    ax.imshow(e, cmap='gray')
    ax.set_title(labels_sum[col], fontsize=10)
    ax.axis('off')

# Row 3: Advanced combinations
ax_c1 = fig.add_subplot(gs[2, 0])
e_noise = feature.canny(img_noisy, sigma=2.0, low_threshold=30/255, high_threshold=90/255)
ax_c1.imshow(e_noise, cmap='gray')
ax_c1.set_title('Canny trên ảnh nhiễu\n(σ=2.0)', fontsize=10)
ax_c1.axis('off')

ax_c2 = fig.add_subplot(gs[2, 1])
e_lc = feature.canny(img_low_contrast, sigma=1.0, low_threshold=10/255, high_threshold=30/255)
ax_c2.imshow(e_lc, cmap='gray')
ax_c2.set_title('Canny – Tương phản thấp\n(ngưỡng thấp hơn)', fontsize=10)
ax_c2.axis('off')

ax_c3 = fig.add_subplot(gs[2, 2])
ax_c3.imshow(img_contour)
ax_c3.set_title('Canny + Contour Detection', fontsize=10)
ax_c3.axis('off')

ax_c4 = fig.add_subplot(gs[2, 3])
ax_c4.imshow(edges_color_or, cmap='gray')
ax_c4.set_title('Color Canny (OR 3 kênh)', fontsize=10)
ax_c4.axis('off')

fig.suptitle('📊 TỔNG KẾT: Canny Edge Detection', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('output_summary.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Hoàn thành notebook Canny Edge Detection!")